# 🥗 Calorie Lens
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shalev396/ml-lab/blob/main/calorie-lens/training/notebook.ipynb)

A photo of a meal -> **every ingredient and its grams -> the nutrition of each ingredient and of the plate**.
Two models fused into one network, all trained in this notebook:
1. **Stage 1, IngredientNet**: an image backbone and one learned query per Nutrition5k ingredient. Its last
   layer is 253 neurons, the grams of each ingredient, rescaled so they add up to the estimated plate mass.
2. **Stage 2a, NutritionLayer**: those 253 neurons are the input layer of a frozen per-gram nutrient table
   (Nutrition5k's own labels are exactly grams x this table), so stage 1 trains end to end on nutrition.
3. **Stage 2b, NutriText**: any food name -> nutrients per 100 g, a small transformer trained from scratch on
   USDA FoodData Central. It powers the food calculator.

Two backbones are compared with the same recipe (torchvision ConvNeXt-B vs DINOv2-B with registers); the
validation winner is retrained with an auxiliary depth head. The test split is looked at once, at the end.

| Step | Where to run it | Time |
|---|---|---|
| Dataset build (once, automatic in the first run): stream Nutrition5k (~195 GB), keep 5 frames per video, publish `shalev396/calorie-lens-data` (~8 GB) | the same runtime, before training (needs an `HF_TOKEN` with write access) | ~1-2 h (the probe in 3a prints the estimate) |
| Full training run (NutriText + 3 experiments) | Colab GPU with 40 GB+ (A100 / H100 / G4, bf16; a 22 GB L4 runs out of memory on DINOv2) | 3.1 h measured (peak 30 GB) |
| `SMOKE_TEST=1` sanity run | any machine, CPU is fine | ~5 min |

## 1. Setup
1a finds the project files (clones the repo on a fresh runtime), 1b installs the dependencies from the root `requirements.txt`, 1c picks the `device`.

In [ ]:
# 1a. Project + files — fetch the repo / model code only if they aren't here (e.g. a fresh Colab runtime)
import os, subprocess
from pathlib import Path

SLUG = "calorie-lens"
if not Path("src").is_dir():                      # not inside <slug>/training yet -> get the repo
    if not Path("ml-lab").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/shalev396/ml-lab.git"], check=True)
    os.chdir(f"ml-lab/{SLUG}/training")
if not Path("../model/model.py").exists():   # model/ is a git submodule (HF repo): code only, skip weights
    subprocess.run(["git", "submodule", "update", "--init", "--depth", "1", f"{SLUG}/model"],
                   cwd="../..", check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
print("working dir:", Path.cwd())

In [ ]:
# 1b. Dependencies — the root requirements.txt is the single source of truth for every project.
# Colab/Jupyter never install a repo's requirements on their own, so this cell does it (fast no-op when satisfied).
%pip install -q -r ../../requirements.txt

In [ ]:
# 1c. Device — whatever this machine has (cuda -> mps -> cpu); nothing platform-specific is installed.
import torch
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

In [ ]:
# 1d. Project code: src/ (training) + ../model/model.py (imported as `model`, the deployed code)
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from src import data_setup, engine, export, model_builder, nutritext, utils
from src.config import Config
import model as M

print({**utils.lib_versions("torch", "torchvision", "transformers", "tokenizers", "numpy", "huggingface_hub"),
       "device": utils.device_name("torch")})

## 2. Config
Every hyperparameter lives in `src/config.py`.
- `SMOKE_TEST=1` runs the whole pipeline on a 6-dish mini build with a tiny random ConvNeXt at 96x128 px and
  exports to `outputs/smoke/model`, never to `../model`.
- Section 3a builds and publishes the dataset by itself while `shalev396/calorie-lens-data` is not on the Hub
  yet (an interrupted build resumes); afterwards runs download it. `BUILD_DATA=1` forces a build.
- `PUSH_TO_HUB = True` uploads the exported folder at the end (full runs only; token from `hf auth login` or an
  `HF_TOKEN` secret / env var).
- `MICRO_BATCH` / `DATA_WORKERS` env vars override the per-step batch and the loader workers.
- Every training line and the traceback of any failed cell are also appended to `outputs/run/train.log`, so
  they survive cleared outputs and kernel restarts (`!tail -50 outputs/run/train.log`).

In [ ]:
os.environ.setdefault("SMOKE_TEST", "0")   # "1" = few-minute sanity run
os.environ.setdefault("BUILD_DATA", "0")   # "1" = force the dataset build (it runs anyway until it is published)
PUSH_TO_HUB = False

cfg = Config()
utils.set_seeds(cfg.seed, "torch")
dev = torch.device(device)
if dev.type == "cuda":                      # TF32 matmuls + cuDNN autotuning: free speed on Ampere and newer
    torch.backends.cuda.matmul.allow_tf32 = torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
plots = cfg.run_dir / "assets"
log = engine.file_log(cfg.run_dir / "train.log")
log(f"session start on {utils.device_name('torch')} | smoke={cfg.smoke} | push={PUSH_TO_HUB}")
print("export to:", cfg.model_dir.relative_to(utils.PROJECT_DIR), "| data:", cfg.data_dir.relative_to(utils.PROJECT_DIR))
pd.Series({k: v for k, v in cfg.to_dict().items() if k != "experiments"}).astype(str).to_frame("value")

## 3. Data
**3a. Build (once: automatic while the dataset is not on the Hub, or `BUILD_DATA=1`).** [Nutrition5k](https://github.com/google-research-datasets/Nutrition5k)
(Google, CC BY 4.0): ~5k cafeteria dishes, every ingredient weighed, an overhead RGB-D scan and four
side-angle videos per dish. `src/n5k_labels.py` fixes the label bugs (Cafe 2 totals logged as 0, x1000 scale
errors, empty names, duplicate table rows) and groups re-scans of one plate (plates were scanned after every
added item) so validation never shares a plate with training; test = the official split.
`src/dataset_build.py` streams every scan and video, keeps 5 frames per camera, and publishes the shards with
the label tables, the USDA FoodData Central table (CC0) and the reviewed ingredient -> USDA map
(`training/labels/ingredient_fdc_map.csv`) as `shalev396/calorie-lens-data`.

**3b. Load.** The published dataset is downloaded into `training/data/`, converted once to memory-mapped
Arrow files, and served as uint8 canvases (378 x 504, aspect kept); augmentation runs on the GPU.

In [ ]:
# 3a. Build the dataset — if it is not published yet or BUILD_DATA=1 (smoke: 6 dishes into outputs/smoke/data, nothing uploaded)
if cfg.build_data or not (cfg.smoke or data_setup.published(cfg.dataset_id)):
    from src import dataset_build, n5k_labels, usda
    root = n5k_labels.download_metadata(cfg.raw_dir / "nutrition5k")
    labels_raw = n5k_labels.load_labels(root)
    display(pd.Series(n5k_labels.check(labels_raw, root)).astype(str).to_frame("labels (V1 checks passed)"))
    imagery = dataset_build.list_imagery()
    dishes = dataset_build.smoke_dishes(labels_raw, imagery) if cfg.smoke else None
    if not cfg.smoke:
        print("probe:", dataset_build.probe(imagery))
    print(dataset_build.build_images(cfg.data_dir, labels_raw, imagery, dishes=dishes,
                                     repo_id=None if cfg.smoke else cfg.dataset_id))
    zips = usda.download(cfg.raw_dir / "usda", usda.GENERIC if cfg.smoke else tuple(usda.RELEASES))
    fdc_map = pd.read_csv(utils.TRAINING_DIR / "labels" / "ingredient_fdc_map.csv")
    tables = dataset_build.write_tables(cfg.data_dir, labels_raw, usda.build_table(zips), fdc_map)
    card = dataset_build.write_card(cfg.data_dir, labels_raw, cfg.dataset_id)
    if not cfg.smoke:
        dataset_build.publish(cfg.data_dir, cfg.dataset_id, tables + [card], "Add label tables, USDA table and card")
else:
    print("using the published dataset:", cfg.dataset_id)

In [ ]:
# 3b. Loaders: train = mixture sampler (half overhead scans, half side frames), val / test = every image
micro = model_builder.pick_micro_batch(cfg, dev)
loaders = data_setup.build_loaders(cfg, dev, micro)
labels = loaders.labels
print(f"micro-batch {micro} x {max(1, cfg.batch_size // micro)} accumulation = batch {cfg.batch_size}")
pd.DataFrame({k: v for k, v in data_setup.summary(loaders).items() if isinstance(v, dict)})

In [ ]:
# A few augmented training images with their labels
batch = next(iter(loaders.train))
x, _ = data_setup.gpu_augment({k: v.to(dev) if torch.is_tensor(v) else v for k, v in batch.items()}, True,
                              torch.Generator(dev).manual_seed(0))
fig, axes = plt.subplots(1, min(4, len(x)), figsize=(14, 3))
for ax, img, d, ov in zip(np.atleast_1d(axes), x.cpu(), batch["dish"], batch["overhead"]):
    kcal, mass = labels.totals[d][0], labels.totals[d][1]
    ax.imshow(img.permute(1, 2, 0)); ax.axis("off")
    ax.set_title(f"{'overhead' if ov else 'side'}: {kcal:.0f} kcal, {mass:.0f} g", fontsize=9)
plt.show()

## 4. Load model
`model.CalorieLens` = backbone -> query decoder (one query per ingredient: presence + grams) + a totals head ->
the 253 gram neurons -> the frozen `NutritionLayer`. The output biases start at the training data's
ingredient frequencies, median grams and mean totals. The throughput probe times the data loader and a full
training step on this GPU (the loader should deliver at least 1.5x what the model consumes).

In [ ]:
probe = {}
for spec in cfg.experiments:
    ecfg = cfg.for_experiment(spec)
    net = model_builder.build_model(ecfg, labels)
    display(pd.Series(export.params_breakdown(net)).map("{:,}".format).to_frame(spec["name"]))
    probe[spec["name"]] = engine.throughput(model_builder.TrainNet(net, False, ecfg.d_model), loaders, ecfg, dev,
                                            steps=2 if cfg.smoke else 20)
    del net
    if dev.type == "cuda":
        torch.cuda.empty_cache()
pd.DataFrame(probe).T

## 5. Training
**5a. NutriText (stage 2b)** on ~470k USDA food names (generic foods + their alternative names; branded
products count 0.3), split by name group so near-identical names never straddle train and test. Compared with
a TF-IDF nearest-neighbour baseline and the mean, and tested zero-shot on the 253 Nutrition5k ingredient names.

**5b. Backbones (stages 1 + 2a, end to end).** Each experiment: AdamW with layer-wise learning-rate decay,
warmup + cosine, bf16, EMA, early stopping on the validation score (mean of the overhead and side-dish calorie
% errors). `last.pt` is written every epoch; re-running a cell after a disconnect resumes it.

**5c. Depth.** The winner (a ConvNeXt within the paired bootstrap CI of a DINOv2 wins the tie: cheaper on the
Space's CPU fallback, no extra licence) is retrained with an auxiliary depth head on the overhead scans.

In [ ]:
# 5a. NutriText
usda_table = pd.read_parquet(cfg.data_dir / "usda.parquet")
nt_data = nutritext.prepare(usda_table, cfg.nt_branded_weight, max_rows=6000 if cfg.smoke else None)
tokenizer = nutritext.train_tokenizer(nt_data.loc[nt_data["split"] == "train", "name"], cfg.nt_vocab)
nt = nutritext.train_nutritext(cfg, tokenizer, nt_data, dev, log=log)
nt_eval = {"test": nutritext.evaluate_all(nt["net"], tokenizer, nt_data, dev),
           "zero_shot_n5k": nutritext.zero_shot_n5k(nt["net"], tokenizer, labels.ingredients, nt_data, dev),
           "best_epoch": nt["best_epoch"], "seconds": nt["seconds"], "rows": nt_data["split"].value_counts().to_dict()}
for part, res in nt_eval["test"].items():
    display(pd.DataFrame({k: v for k, v in res.items() if k != "n"}).round(3).rename_axis(f"{part} test ({res['n']} names)"))
pd.DataFrame(nt_eval["zero_shot_n5k"]).round(1).rename_axis("Nutrition5k names, per 100 g")

In [ ]:
# 5b. One run per backbone
results, val_preds = {}, {}
for spec in cfg.experiments:
    ecfg = cfg.for_experiment(spec)
    tnet = model_builder.TrainNet(model_builder.build_model(ecfg, labels), ecfg.depth_aux, ecfg.d_model)
    results[spec["name"]] = {**engine.train(tnet, loaders, ecfg, dev, spec["name"], log=log), "spec": spec}
    val_preds[spec["name"]] = engine.evaluate(results[spec["name"]]["net"].to(dev), loaders.val, dev)
    results[spec["name"]]["net"].cpu()
    del tnet
    if dev.type == "cuda":
        torch.cuda.empty_cache()
winner, reason = engine.pick_winner(results, val_preds, labels)
log(f"backbone winner: {winner} - {reason}")

In [ ]:
# 5c. The winner + auxiliary depth head (its own loaders: overhead scans now return their depth maps)
spec3 = {**results[winner]["spec"], "name": f"E3 {winner.split(' ', 1)[1]} + depth", "depth_aux": True}
ecfg3 = cfg.for_experiment(spec3)
loaders3 = data_setup.build_loaders(ecfg3, dev, micro)
tnet = model_builder.TrainNet(model_builder.build_model(ecfg3, labels), True, ecfg3.d_model)
results[spec3["name"]] = {**engine.train(tnet, loaders3, ecfg3, dev, spec3["name"], log=log), "spec": spec3}
val_preds[spec3["name"]] = engine.evaluate(results[spec3["name"]]["net"].to(dev), loaders.val, dev)
del tnet, loaders3
final_name = spec3["name"] if results[spec3["name"]]["val_score"] < results[winner]["val_score"] else winner
log(f"deployed: {final_name} {({k: round(r['val_score'], 2) for k, r in results.items()})}")
display(Image(str(export.plot_histories({k: r["history"] for k, r in results.items()}, plots / "training_curves.png")), width=900))

## 6. Evaluation
On validation: the mass blend (direct mass head vs the sum of the ingredient grams) and the display threshold
are fitted. Then the test split, once: **overhead scans** (the official `depth_test` split; the headline,
comparable with published Nutrition5k results), side frames and side dishes (all frames averaged),
`test_clean` (plates never seen in training) and the original, uncorrected labels. MAE and percent MAE
(MAE / mean of the truth). The ablation shows what each part of the fused model is worth.

In [ ]:
final = results[final_name]["net"].to(dev)
blend, blend_scores = engine.fit_mass_blend(val_preds[final_name], labels)
threshold = engine.fit_presence_threshold(val_preds[final_name], labels)
test_preds = engine.evaluate(final, loaders.test, dev)
clean = np.isin(test_preds.dish, np.flatnonzero(labels.test_clean))
test = engine.summarize(test_preds, labels, blend, threshold)
test_clean = engine.summarize(test_preds.subset(clean), labels, blend, threshold)
test_original = engine.summarize(test_preds, labels, blend, threshold, original=True)
ci = engine.bootstrap_ci(test_preds, labels, blend)
ablation = engine.ablation(test_preds, labels, blend)
print(f"mass blend w={blend}, display threshold {threshold}, overhead kcal PMAE 95% CI [{ci[0]:.2f}, {ci[1]:.2f}]")
cols = [f"{t}_{m}" for t in M.TOTALS for m in ("mae", "pmae")]
display(pd.DataFrame({f"test {v}": m for v, m in test.items()} | {f"clean {v}": m for v, m in test_clean.items()}
                     | {f"original labels {v}": m for v, m in test_original.items()}).T[["n", *cols]].round(2))
pd.Series(ablation).round(2).to_frame("overhead test kcal % error")

In [ ]:
# Every experiment on the test split (selection was on validation; reported for the Experiments table)
experiments = {}
for name, r in results.items():
    p = test_preds if name == final_name else engine.evaluate(r["net"].to(dev), loaders.test, dev)
    w = blend if name == final_name else engine.fit_mass_blend(val_preds[name], labels)[0]
    experiments[name] = engine.experiment_row(r, engine.summarize(p, labels, w), export.params_breakdown(r["net"])["total"])
    r["net"].cpu()
display(pd.DataFrame(experiments).T)
over = test_preds.subset(test_preds.overhead)
pred_kcal = engine.predicted_targets(over, labels.densities, "reconciled", blend)[:, 0]
for path in (export.plot_bars({k: v["test_kcal_pmae"] for k, v in experiments.items()}, final_name, plots / "experiments.png",
                              "Test (overhead): calorie % error per experiment", "kcal percent MAE (lower is better)"),
             export.plot_bars(ablation, list(ablation)[-1], plots / "ablation.png",
                              "Test (overhead): what each part of the fused model is worth", "kcal percent MAE"),
             export.plot_scatter(pred_kcal, labels.totals[over.dish, 0], plots / "kcal_scatter.png",
                                 f"Test (overhead): predicted vs true kcal, {len(over.dish)} dishes"),
             export.plot_error_by_bin(pred_kcal, labels.totals[over.dish, 0], plots / "error_by_kcal.png"),
             export.plot_nutritext(nt_eval["test"], plots / "nutritext.png")):
    display(Image(str(path), width=760))

## 7. Inference
The deployed model = the winning stage-1 network + NutriText + the fitted blend and threshold, saved and loaded
back exactly as the Space and the Inference Endpoint do (`model.load(dir, device)`), then run on example photos
from the test split (`predict`, `analyze` with the ingredient heatmaps) and on a food list (`calculate`).

In [ ]:
deployed = export.assemble(final.cpu(), nt, blend, threshold)
best_dir = export.save_model(deployed, tokenizer, cfg.run_dir / "best")
predictor = M.load(best_dir, device)
examples = sorted((utils.SPACE_DIR / "examples").glob("*.jpg")) or data_setup.save_examples(cfg, cfg.run_dir / "examples")
for path in examples:
    r = predictor.predict(path)
    print(f"{path.name:32s} {r['totals']['kcal']:7.1f} kcal {r['totals']['mass_g']:6.1f} g  <- " +
          ", ".join(f"{i['name']} {i['grams']:.0f} g" for i in r["ingredients"][:4]))
# the loaded predictor reproduces the in-memory model
x = M.preprocess(M.to_pil(examples[0]), deployed.image_size)[None].float() / 255
with torch.no_grad():
    assert torch.allclose(deployed(x)["totals"], predictor.model.cpu()(x)["totals"], atol=1e-4)
predictor.model.to(dev)
display(pd.DataFrame(predictor.calculate([("grilled salmon", 150), ("white rice", 200), ("steamed broccoli", 80)])["items"]))
display(Image(str(export.save_heatmap_example(predictor, examples[0], plots / "heatmaps.png")), width=900))

## 8. Export
Writes `model.safetensors` + `config.json` + `nutritext_tokenizer.json`, `metrics.json` (deployed model, every
experiment, NutriText, ablation, both label versions), the graphs in `assets/`, `NOTICE` + the Apache-2.0 text
if the shipped backbone is DINOv2, and refreshes the model card. `PUSH_TO_HUB = True` then uploads the folder
to `shalev396/calorie-lens`.

In [ ]:
metrics = export.build_metrics(
    headline=test["overhead"], test=test, test_clean=test_clean, test_original=test_original, ci=ci,
    experiments=experiments, nutritext=nt_eval, ablation=ablation,
    mass_blend={"value": blend, "val_scores": blend_scores}, threshold=threshold, winner=final_name,
    data=data_setup.summary(loaders), params=export.params_breakdown(deployed),
    train_time_s=sum(r["seconds"] for r in results.values()) + nt["seconds"], device=utils.device_name("torch"),
    smoke=cfg.smoke, selection={"backbone_winner": winner, "reason": reason}, throughput=probe,
    histories={k: r["history"] for k, r in results.items()})
export.export(deployed, tokenizer, metrics, cfg.model_dir, plots)
pd.Series(metrics["metrics"]).to_frame("test")

In [ ]:
if PUSH_TO_HUB and not cfg.smoke:
    print(utils.upload_model(cfg.model_dir))
else:
    print("not uploading (PUSH_TO_HUB is False or this is a smoke run)")